## Getting started

We build a simple strategy that goes long when overnight return is negative and exits by the end of the day. If price moves against us, we use a stop loss to get out. We have 1 minute bars of AAPL stock prices

In [ ]:
# %%checkall
import polars as pl
import gambit as pq
import numpy as np

# read 1 minute price bars
aapl_file = pq.find_in_subdir('.', 'AAPL.csv.gz')
aapl = pl.read_csv(aapl_file, try_parse_dates=True).select('timestamp', 'c')
# the date corresponding to each 1minute timestamp
aapl = aapl.with_columns(pl.col('timestamp').dt.date().alias('date')) 
# compute overnight return
aapl = aapl.with_columns(pl.when(pl.col('date') > pl.col('date').shift(1)).then(pl.col('c') / pl.col('c').shift(1) - 1).otherwise(None).alias('overnight_ret'))

aapl = aapl.with_columns(pl.when(pl.col('date') > pl.col('date').shift(1)).then(pl.col('c') / pl.col('c').shift(1) - 1).otherwise(None).alias('overnight_ret'))
aapl = aapl.with_columns((pl.col('overnight_ret') < 0).alias('overnight_ret_negative'))  # whether overnight return is negative
# mark points just before EOD. We enter a marker order at these points so we have one bar to get filled
aapl = aapl.with_columns((pl.col('date').shift(-2) > pl.col('date')).fill_null(False).alias('eod'))   
# if the price drops by 1% after we enter in the morning take our loss and get out
aapl = aapl.with_columns(pl.when(pl.col('date') > pl.col('date').shift(1)).then(pl.col('c')).otherwise(None).forward_fill().alias('bod_price'))
aapl = aapl.with_columns(pl.lit('AAPL').alias('symbol'))
stop_return_func = pq.PriceFuncArrays(aapl['symbol'].to_numpy(), aapl['timestamp'].to_numpy(), np.full(len(aapl), -0.1))
aapl = aapl.with_columns((pl.col('c') < pl.col('bod_price') * 0.99).alias('stop'))

strat_builder = pq.StrategyBuilder(data=aapl)   
strat_builder.add_contract('AAPL')
# convert timestamps from nanoseconds (pandas convention) to minutes so they are easier to view
timestamps = aapl['timestamp'].to_numpy().astype('M8[m]')  
prices = aapl['c'].to_numpy()
# create a dictionary from contract name=>timestamp => price for use in the price function
price_dict = {'AAPL': {timestamps[i]: prices[i] for i in range(len(timestamps))}}
# create the price function that the strategy will use for looking up prices 
price_function = pq.PriceFuncDict(price_dict=price_dict)
strat_builder.set_price_function(price_function)

# FiniteRiskEntryRule allows us to enter trades and get out with a limited loss when a stop is hit.
# This enters market orders, if you want to use limit orders, set the limit_increment argument
entry_rule = pq.BracketOrderEntryRule(
    reason_code='POS_OVERNIGHT_RETURN',  # this is useful to know why we entered a trade
    price_func=price_function, 
    long=True,  # whether we enter a long or short position
    percent_of_equity=0.1,  # set the position size so that if the stop is hit, we lose no more than this
    # stop price is used for position sizing.  Also, we will not enter if the price is already below 
    # stop price for long trades and vice versa
    stop_return_func=stop_return_func,
    single_entry_per_day=True)  # if we are stopped out, do we allow re-entry later in the day

# ClosePositionExitRule fully exits a position using either a market or limit order
# In this case, we want to exit at EOD so we are flat overnight
exit_rule_stop = pq.ClosePositionExitRule(   
    reason_code='STOPPED_OUT',
    price_func=price_function)

# Exit when the stop price is reached
exit_rule_eod = pq.ClosePositionExitRule(
    reason_code='EOD',
    price_func=price_function)

# Setup the rules we setup above so they are only called when the columns below in our data dataframe are true
# position filters allow you to choose when the rule runs, "zero" orders it to run only when 
# we don't have a current position, positive and negative similarly run the rule when we
# are currently long or short respectively
strat_builder.add_series_rule('overnight_ret_negative', entry_rule, position_filter='zero')
strat_builder.add_series_rule('eod', exit_rule_eod, position_filter='positive')
strat_builder.add_series_rule('stop', exit_rule_stop, position_filter='positive')

# create the strategy and run it
strategy = strat_builder()
strategy.run()

In [ ]:
# Lets evaluate how the strategy did
strategy.df_roundtrip_trades()  

In [ ]:
# Lets evaluate how the strategy did
metrics = strategy.evaluate_returns(plot=pq.has_display())